# Random Forest: Baseline, Grid Search, Feature Engineering, RMSE
**Model**: RandomForestRegressor. **Target**: price. **Primary metric**: RMSE.

**Validation strategy:** shared 80/20 hold-out (`random_state=42`) + 5-fold CV on the training split for tuning; test data is not used during development.

**Rimma's Steps/Workflow**: 
1. Use Dariya's shared preprocessing, apply model-specific preprocessing for Random Forest
2. Develop baseline model
3. Improve through tuning and feature engineering
4. Evaluate metrics and model performance (overfitting, generalization, RMSE)
5. Form conclusions (performance, strengths/limitations)

## 1a. Load data and shared split
Same feature set and split as Dariya's 01_data_preprocessing.ipynb (80/20, `random_state=42`) so RMSE is comparable across the team's models.

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

train_df = pd.read_csv("../data/challenge1_train.csv")
test_df = pd.read_csv("../data/challenge1_test.csv")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7398, 10)
Test shape: (2467, 9)


In [2]:
X = train_df.drop(columns=["price", "ID"])
y = train_df["price"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print("X_train:", X_train.shape, "| X_val:", X_val.shape)

X_train: (5918, 8) | X_val: (1480, 8)


## 1b. Model-specific preprocessing

The shared preprocessor applies StandardScaler to numeric features and one-hot encodes categoricals. For Random Forest:

- Scaling is unnecessary
- One-hot encoding is kept for `category`, `has_photo` and `state`, since scikit-learn trees need numeric input. `handle_unknown="ignore"` is kept as the test set contains the state WY, which never appears in the training data.
- The preprocessor sits inside a Pipeline, so inside cross-validation it is fit only on each training fold (no leakage).

In [3]:
numeric_features = ["bathrooms", "bedrooms", "square_feet", "latitude", "longitude"]
categorical_features = ["category", "has_photo", "state"]

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

In [4]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Random Forest (Baseline, default hyperparameters)

In [5]:
rf_base = Pipeline([
    ("prep", rf_preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

rf_base.fit(X_train, y_train)

cv_scores = -cross_val_score(rf_base, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error", n_jobs=-1)

baseline = {
    "Train RMSE": rmse(y_train, rf_base.predict(X_train)),
    "CV RMSE": cv_scores.mean(),
    "Validation RMSE": rmse(y_val, rf_base.predict(X_val)),
}
print({k: round(float(v), 2) for k, v in baseline.items()})

{'Train RMSE': 159.97, 'CV RMSE': 424.34, 'Validation RMSE': 456.55}


## 3a. Grid Search
Hyperparameters and why they were chosen:

| Hyperparameter | Values | Reason |
|---|---|---|
| `n_estimators` | 200, 400 | More trees lower variance, with diminishing returns and higher cost |
| `max_depth` | 20, None | Limits tree complexity (overfitting control) |
| `min_samples_leaf` | 1, 3, 5 | Larger leaves smooth predictions and reduce noise-fitting |
| `max_features` | 0.33, 0.6, 1.0 | Fraction of features per split; lower values decorrelate trees |

In [6]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [20, None],
    "model__min_samples_leaf": [1, 3, 5],
    "model__max_features": [0.33, 0.6, 1.0],
}

grid = GridSearchCV(
    estimator=Pipeline([
        ("prep", rf_preprocessor),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=1,
)

t0 = time.time()
grid.fit(X_train, y_train)
print(f"Grid Search time: {(time.time() - t0)/60:.1f} min")
print("Best params:", grid.best_params_)
print(f"Best CV RMSE: {-grid.best_score_:,.2f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Grid Search time: 2.0 min
Best params: {'model__max_depth': 20, 'model__max_features': 0.33, 'model__min_samples_leaf': 1, 'model__n_estimators': 400}
Best CV RMSE: 412.69


In [7]:
results = (pd.DataFrame(grid.cv_results_)
           .assign(cv_rmse=lambda d: -d["mean_test_score"],
                   train_rmse=lambda d: -d["mean_train_score"])
           [["param_model__n_estimators", "param_model__max_depth",
             "param_model__min_samples_leaf", "param_model__max_features",
             "cv_rmse", "train_rmse"]]
           .sort_values("cv_rmse"))
results.columns = ["n_estimators", "max_depth", "min_samples_leaf", "max_features", "cv_rmse", "train_rmse"]
results.head(10).round(2)

,n_estimators,max_depth,min_samples_leaf,max_features,cv_rmse,train_rmse
1,400,20,1,0.33,412.69,181.23
0,200,20,1,0.33,413.39,182.60
27,400,None,3,0.60,414.80,285.99
19,400,None,1,0.33,415.10,155.82
25,400,None,1,0.60,415.13,155.51
9,400,20,3,0.60,415.37,288.78
7,400,20,1,0.60,415.48,171.44
24,200,None,1,0.60,415.59,156.68
26,200,None,3,0.60,415.74,286.50
6,200,20,1,0.60,415.85,172.30


### Compare (Baseline vs. Tuned)

In [8]:
best_rf = grid.best_estimator_

tuned = {
    "Train RMSE": rmse(y_train, best_rf.predict(X_train)),
    "CV RMSE": -grid.best_score_,
    "Validation RMSE": rmse(y_val, best_rf.predict(X_val)),
}

summary = pd.DataFrame({
    "Train RMSE": [baseline["Train RMSE"], tuned["Train RMSE"]],
    "CV RMSE": [baseline["CV RMSE"], tuned["CV RMSE"]],
    "Validation RMSE": [baseline["Validation RMSE"], tuned["Validation RMSE"]],
}, index=["RF baseline (default)", "RF tuned (grid search)"]).round(2)
summary["Gap (Val - Train)"] = (summary["Validation RMSE"] - summary["Train RMSE"]).round(2)  

summary[["Train RMSE", "CV RMSE", "Validation RMSE"]]

,Train RMSE,CV RMSE,Validation RMSE
RF baseline (default),159.97,424.34,456.55
RF tuned (grid search),181.80,412.69,444.87


## 3b. Feature Engineering
Four new columns are computed from each row's own values only (no information from other rows or the validation set, so no leakage):

- `sqft_per_bedroom` and `sqft_per_bathroom`: space per room
- `total_rooms`: bedrooms plus bathrooms
- `lat_lon_interaction`: latitude × longitude, a rough location interaction

The hyperparameters found by the grid search are reused and the same 5 CV folds are used, so changes in the RMSE come from the new features.

In [9]:
def add_features(df):
    df = df.copy()
    df["sqft_per_bedroom"] = df["square_feet"] / (df["bedrooms"] + 1)   
    df["sqft_per_bathroom"] = df["square_feet"] / df["bathrooms"].clip(lower=0.5)
    df["total_rooms"] = df["bedrooms"] + df["bathrooms"]
    df["lat_lon_interaction"] = df["latitude"] * df["longitude"]
    return df

eng_numeric = numeric_features + ["sqft_per_bedroom", "sqft_per_bathroom", "total_rooms", "lat_lon_interaction"]

eng_preprocessor = ColumnTransformer([
    ("num", "passthrough", eng_numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}

rf_fe = Pipeline([
    ("prep", eng_preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **best_params)),
])

X_train_fe, X_val_fe = add_features(X_train), add_features(X_val)

cv_scores_fe = -cross_val_score(rf_fe, X_train_fe, y_train, cv=cv,
                                scoring="neg_root_mean_squared_error", n_jobs=-1)
rf_fe.fit(X_train_fe, y_train)

fe = {
    "Train RMSE": rmse(y_train, rf_fe.predict(X_train_fe)),
    "CV RMSE": cv_scores_fe.mean(),
    "Validation RMSE": rmse(y_val, rf_fe.predict(X_val_fe)),
}
print({k: round(float(v), 2) for k, v in fe.items()})

{'Train RMSE': 173.78, 'CV RMSE': 417.11, 'Validation RMSE': 450.87}


## 4a. Overfitting Check

In [10]:
overfit = pd.concat([
    summary[["Train RMSE", "Validation RMSE"]],
    pd.DataFrame([fe], index=["RF tuned + engineered features"])[["Train RMSE", "Validation RMSE"]].round(2),
])
overfit["Gap (Val - Train)"] = (overfit["Validation RMSE"] - overfit["Train RMSE"]).round(2)
overfit

,Train RMSE,Validation RMSE,Gap (Val - Train)
RF baseline (default),159.97,456.55,296.58
RF tuned (grid search),181.80,444.87,263.07
RF tuned + engineered features,173.78,450.87,277.09


**All three models overfit**. (They are **not underfitting**; underfitting would show high error on the training data as well, but here the training error is low.)
- Their training RMSE is far lower than their validation RMSE, so each one fits the training data much better than new data.
- The baseline Random Forest has the largest gap (296.58), meaning it memorizes the training data the most.
- Tuning raised training RMSE slightly and lowered validation RMSE, which narrowed the gap to 263.07, but the tuned model still overfits.
- The engineered features widened the gap again to 277.09, so they **did not improve generalization**.

## 4b. Final Comparison

In [11]:
final_summary = pd.concat([
    summary.drop(columns="Gap (Val - Train)"),
    pd.DataFrame([fe], index=["RF tuned + engineered features"]),
]).round(2)

final_summary["Gap (Val - Train)"] = (final_summary["Validation RMSE"] - final_summary["Train RMSE"]).round(2)
final_summary["CV change vs prev. row"] = final_summary["CV RMSE"].diff().round(2)
final_summary["Val change vs prev. row"] = final_summary["Validation RMSE"].diff().round(2)

final_summary

,Train RMSE,CV RMSE,Validation RMSE,Gap (Val - Train),CV change vs prev. row,Val change vs prev. row
RF baseline (default),159.97,424.34,456.55,296.58,NaN,NaN
RF tuned (grid search),181.80,412.69,444.87,263.07,-11.65,-11.68
RF tuned + engineered features,173.78,417.11,450.87,277.09,4.42,6.00


## 5. Conclusions

The baseline Random Forest model gave a reasonable starting point but overfit heavily. Grid search improved performance modestly and reduced the overfitting a little, but the tuned model still overfits. Feature engineering did not help and slightly worsened results. The tuned Random Forest on the original features had the lowest CV and val RMSE, as well as the lowest val/train gap. Because the gains from tuning were small and the top grid-search results were within about 3 points of each other in CV RMSE, the Random Forest seems close to its limit on these features, so further tuning is unlikely to improve performance much.

In all, the Random Forest is less interpretable than a single Decision Tree, but it is typically more stable and can be a strong general-purpose model without extensive tuning. These metrics should be compared with the rest of the team's models to determine which should be chosen for the final predictions.